# QB Motion Atlas — V2 Embedding Training (task 104)

Trains the per-phase embedding model (`docs/embedding_methodology.md`, `models/embedding_net.py`) on the real reference dataset, once one exists.

**Status as of 2026-09-25:** this notebook is ready to run but there is no real reference data to run it on yet — this cloud dev environment's network policy blocks `youtube.com`, so the seed clips' actual pose/feature data can't be computed (see `docs/research_log.md`). Every function this notebook calls is already unit-tested against synthetic data in `tests/`; this notebook is for the real training run once real data exists, not another synthetic demo.

**How to get data into this notebook:**
1. Locally, with the real reference clips processed and `qb_reference_features` populated with per-phase rows (task 116): `python -m db.export_reference_features data/reference_features_export.json`.
2. Upload that JSON file here (Colab: the file browser on the left; Kaggle: Add Data → Upload).
3. Run all cells below.


In [ ]:
# If running on Colab/Kaggle, install this repo's dependencies that aren't preinstalled.
# (torch is preinstalled on both platforms; this is a no-op there.)
import importlib.util
if importlib.util.find_spec("torch") is None:
    %pip install torch


In [ ]:
import json
import random
import sys
from pathlib import Path

# On Colab/Kaggle this repo won't be on sys.path by default -- clone it first:
#   !git clone https://github.com/Sithranjan-Suresh/QB-Motion-Atlas.git
#   sys.path.insert(0, "QB-Motion-Atlas")
# Locally (running from the repo root) this is already true.
REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from pipeline.embedding.dataset_split import split_clips
from pipeline.embedding.sampling import PhaseRecord, generate_triplets
from models.train_embedding import train_embedding_net
from eval.retrieval_accuracy import LabeledVector, leave_one_out_retrieval_accuracy
from eval.discriminative_validity import compute_discriminative_validity


## 1. Load the exported reference data

In [ ]:
EXPORT_PATH = "reference_features_export.json"  # uploaded per the instructions above

with open(EXPORT_PATH) as f:
    raw_records = json.load(f)

records = [
    PhaseRecord(qb_name=r["qb_name"], clip_id=r["clip_id"], phase_name=r["phase_name"], feature_vector=r["feature_vector"])
    for r in raw_records
]
print(f"Loaded {len(records)} (clip, phase) records across {len(set(r.qb_name for r in records))} QBs")


## 2. Split at the clip level (task 101)

Splitting happens on clip_ids, not on the PhaseRecord list directly, so all six phases of a held-out clip stay together in the same split (`docs/embedding_methodology.md`'s no-leakage rule).

In [ ]:
clip_ids_by_qb = {qb: sorted({r.clip_id for r in records if r.qb_name == qb}) for qb in {r.qb_name for r in records}}

split = split_clips(clip_ids_by_qb, rng=random.Random(0))
print(f"train: {len(split.train)} clips, val: {len(split.val)} clips, test: {len(split.test)} clips")

train_records = [r for r in records if r.clip_id in set(split.train)]
val_records = [r for r in records if r.clip_id in set(split.val)]
test_records = [r for r in records if r.clip_id in set(split.test)]


## 3. Pick a phase, generate triplets, and train (tasks 100, 103, 105)

In [ ]:
PHASE = "release"  # re-run this notebook once per phase -- one EmbeddingNet per phase, per docs/embedding_methodology.md

phase_train_records = [r for r in train_records if r.phase_name == PHASE]
feature_keys = sorted(phase_train_records[0].feature_vector.keys()) if phase_train_records else []

triplets = generate_triplets(phase_train_records, rng=random.Random(0))
print(f"{len(triplets)} training triplets for phase '{PHASE}'")

result = train_embedding_net(triplets, feature_keys, epochs=200)
model, loss_history = result.model, result.loss_history


In [ ]:
import matplotlib.pyplot as plt

plt.plot(loss_history)
plt.xlabel("epoch")
plt.ylabel("triplet loss")
plt.title(f"Training loss -- phase: {PHASE}")
plt.show()
# Log this curve in docs/research_log.md once run for real (task 105).


## 4. Evaluate on the held-out test split (task 106)

In [ ]:
import torch

from models.train_embedding import _vectors_to_tensor  # reusing the exact normalization stats training used

phase_test_records = [r for r in test_records if r.phase_name == PHASE]
if len(phase_test_records) < 2:
    print(f"Not enough test-split records for phase '{PHASE}' to evaluate retrieval accuracy yet "
          f"(only {len(phase_test_records)}) -- needs task 99's dataset expansion.")
else:
    def embedding_similarity(vec_a: dict, vec_b: dict) -> float:
        with torch.no_grad():
            emb_a = model(_vectors_to_tensor([vec_a], feature_keys, result.mean, result.std))
            emb_b = model(_vectors_to_tensor([vec_b], feature_keys, result.mean, result.std))
        return torch.nn.functional.cosine_similarity(emb_a, emb_b).item()

    labeled_test_items = [
        LabeledVector(label=r.qb_name, item_id=r.clip_id, vector=r.feature_vector) for r in phase_test_records
    ]
    accuracy = leave_one_out_retrieval_accuracy(labeled_test_items, embedding_similarity, k_values=(1, 3))
    print(f"Held-out top-1/top-3 retrieval accuracy (phase '{PHASE}'): {accuracy}")
    # Log this number in docs/research_log.md once run for real (task 106).


**Note on small test splits:** with very few clips per QB (this project's realistic near-term size before task 99's expansion), the test split can end up with only one clip for a given QB. Leave-one-out retrieval accuracy for that QB is then structurally 0% -- there is no *other* same-QB item in the candidate pool to retrieve, no matter how good the embedding is. A low number here isn't necessarily a bad embedding; check `len(phase_test_records)` per QB before reading too much into the accuracy number at this dataset size.


## 5. Iterate (task 107)

Re-run section 3 with different `epochs`/`lr`/`EMBEDDING_DIM` (in `models/embedding_net.py`) and re-check section 4's held-out accuracy. Log each iteration's result in `docs/research_log.md` -- keep the test split untouched until the final reported number (task 101's whole point).

## 6. Export to ONNX for production inference (task 108)

In [ ]:
from models.export_onnx import save_checkpoint

CHECKPOINT_DIR = f"embedding_checkpoint_{PHASE}"
save_checkpoint(model, feature_keys, result.mean, result.std, CHECKPOINT_DIR)
print(f"Saved ONNX model + normalization metadata to {CHECKPOINT_DIR}/ -- download this directory and place it "
      f"under models/checkpoints/{PHASE}/ in the repo (see .gitignore's models/checkpoints/ rule) for "
      f"models/embedding_inference.py (task 109) to load.")
